# `Ineq_cycle`: exact siglocal audit and NPA quantum bounds

This notebook studies the single inequality

$$
I_{\mathrm{cycle}}(p)
=\sum_{a,b,c,x,y,z} c_{abcxyz}\,p(abc\mid xyz)\geq 20
$$

for the corrected tripartite siglocal polytope.

It performs three logically separate tasks:

1. **Exact classical audit.** It regenerates the 8128 distinct deterministic
   siglocal vertices and verifies the classical bound $20$ with integer
   arithmetic.
2. **Exact facet audit.** It computes the affine dimension of the full polytope
   and of the saturating face using rational arithmetic.
3. **Quantum bounds.** It evaluates a known explicit quantum realization and
   runs `QuantumNPA.jl` at hierarchy levels 2, 3, and 4 with tight SCS settings.

The notebook is self-contained: it does not read the perturbation-search files
or any earlier notebook.


## A crucial convention about the direction of the bound

This is a **lower-bound inequality**. Define

$$
q_*:=\min_{p\in\mathcal Q} I_{\mathrm{cycle}}(p),
\qquad
v_*:=\max_{p\in\mathcal Q}\bigl(20-I_{\mathrm{cycle}}(p)\bigr)
=20-q_*.
$$

At each NPA level, `npa_min` minimizes over an outer approximation of the
quantum set. It therefore returns a certified **lower bound** $L_k\leq q_*$.
Equivalently,

$$
v_*\leq 20-L_k,
$$

so $20-L_k$ is the requested **NPA upper bound on the quantum violation**.

An explicit feasible quantum realization gives the opposite side of the
bracket: its value $I_{\rm feasible}$ is an upper bound on $q_*$, and
$20-I_{\rm feasible}$ is a lower bound on $v_*$. The final cell reports both
intervals so that the direction cannot be confused.


## How to run

1. Open this file with Jupyter using the Julia kernel.
2. Choose **Kernel → Restart Kernel and Run All Cells**.
3. The first code cell installs only missing packages.
4. The three NPA problems are then solved sequentially. Results are saved after
   every completed level in the directory `ineq_cycle_npa_results` beside the
   notebook.

The main output file is `ineq_cycle_npa_bounds.tsv`.


In [4]:
# Package bootstrap. Existing installations are left untouched; only packages
# missing from the active Julia environment are added.
import Pkg

required_packages = [
    "JuMP",
    "MathOptInterface",
    "QuantumNPA",
    "SCS",
]

missing_packages = [
    package for package in required_packages
    if Base.find_package(package) === nothing
]

if !isempty(missing_packages)
    println("Installing missing Julia packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required Julia packages are already available.")
end


All required Julia packages are already available.


In [5]:
using Dates
using JuMP
using LinearAlgebra
using Printf
using QuantumNPA
using SCS
using SparseArrays

# Import MathOptInterface directly. This avoids the Julia 1.12 error caused by
# trying to access it as `JuMP.MathOptInterface`.
import MathOptInterface as MOI

# -------------------------- USER CONFIGURATION --------------------------

# Carlos used NPA levels 2, 3, and 4. Higher levels are tighter but larger.
NPA_LEVELS = [2, 3, 4]

# Tight SCS settings, identical to the settings used in the precision-audited
# screening notebook that already ran successfully on the user's installation.
SCS_EPS_ABS = 1e-8
SCS_EPS_REL = 1e-8
SCS_MAX_ITERS = 1_000_000

# Small tolerance used only for numerical consistency diagnostics.
NUMERICAL_TOLERANCE = 1e-5

# Run the exact affine-rank calculation needed for the facet test. Keep this
# true for the research run. It is separate from, and does not affect, the NPA.
RUN_EXACT_FACET_AUDIT = true

RESULTS_DIRECTORY = joinpath(pwd(), "ineq_cycle_npa_results")

# ------------------------------------------------------------------------

TIGHT_SCS_SOLVER = optimizer_with_attributes(
    SCS.Optimizer,
    "eps_abs" => SCS_EPS_ABS,
    "eps_rel" => SCS_EPS_REL,
    "max_iters" => SCS_MAX_ITERS,
    "verbose" => 0,
)

mkpath(RESULTS_DIRECTORY)
println("Output directory: ", abspath(RESULTS_DIRECTORY))


Output directory: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\ineq_cycle_npa_results


## 1. Canonical coordinates

The behavior is represented by a 64-component vector. Input blocks are ordered

`xyz = 000, 001, 010, 011, 100, 101, 110, 111`,

and, inside every block, outputs are ordered

`abc = 000, 001, 010, 011, 100, 101, 110, 111`.

Thus $c$ and $z$ are the fastest-changing indices. The function below is the
single source of truth for this convention throughout the notebook.


In [7]:
TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

"""One-based index of p(a,b,c|x,y,z) in the canonical 64-vector."""
prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)

"""Convert an 8x8 (input rows, output columns) table to canonical order."""
function matrix_to_canonical_vector(matrix)
    size(matrix) == (8, 8) || error("Expected an 8x8 coefficient matrix")
    coefficients = Vector{eltype(matrix)}(undef, 64)

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        input_row = 1 + z + 2*y + 4*x
        output_column = 1 + c + 2*b + 4*a
        coefficients[prob_index(a, b, c, x, y, z)] =
            matrix[input_row, output_column]
    end

    return coefficients
end


matrix_to_canonical_vector

## 2. Exact definition of `Ineq_cycle`

Rows of the displayed matrix are input triples $xyz$ and columns are output
triples $abc$, both in lexicographic binary order. All coefficients and the
siglocal bound are integers; no rounding or rational reconstruction occurs.


In [9]:
INEQ_CYCLE_MATRIX = Int[
    0  12  12  15  12  15  15   0;
    0   4  15  12  15  12   5   0;
    0  15   4  12  15   5  12   0;
    8   4   4   0   0   5   5  10;
    0  15  15   5   4  12  12   0;
    8   4   0   5   4   0   5  10;
    8   0   4   5   4   5   0  10;
    5   0   0   4   0   4   4   8
]

INEQ_CYCLE = matrix_to_canonical_vector(INEQ_CYCLE_MATRIX)
SIGLOCAL_BOUND = 20

@assert length(INEQ_CYCLE) == 64
@assert reshape(INEQ_CYCLE, 8, 8)' == INEQ_CYCLE_MATRIX

println("Encoded Ineq_cycle exactly with siglocal bound ", SIGLOCAL_BOUND, ".")


Encoded Ineq_cycle exactly with siglocal bound 20.


## 3. Corrected siglocal vertices

Each party has a binary output and receives its own binary input plus one
neighbor's input. The corrected deterministic dependencies are

- clockwise: $a=f_A(x,z)$, $b=f_B(y,x)$, $c=f_C(z,y)$;
- counterclockwise: $a=f_A(x,y)$, $b=f_B(y,z)$, $c=f_C(z,x)$.

There are $16^3=4096$ deterministic strategies in each orientation. Their
intersection consists of the 64 ordinary local deterministic behaviors, so the
union has $4096+4096-64=8128$ distinct vertices.


In [11]:
"""Evaluate one of the 16 deterministic binary functions of two bits."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one corrected deterministic siglocal behavior."""
function deterministic_vertex(fA::Int, fB::Int, fC::Int, orientation::Symbol)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate all 4096 deterministic vertices of one orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    @assert row == 16^3 + 1
    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64
    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end
    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)
vertices_siglocal = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)
intersection_size = length(intersect(clockwise_set, counterclockwise_set))

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_siglocal)

@assert size(vertices_clockwise) == (4096, 64)
@assert size(vertices_counterclockwise) == (4096, 64)
@assert intersection_size == 64
@assert size(vertices_siglocal) == (8128, 64)

println("Corrected siglocal vertex set")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", intersection_size)
println("  distinct union:            ", size(vertices_siglocal, 1))


Corrected siglocal vertex set
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128


## 4. Exact validity audit

Because every vertex and coefficient is integral, all values below are exact.
For a linear inequality it is sufficient to test the vertices of the polytope.


In [13]:
values_clockwise = vertices_clockwise * INEQ_CYCLE
values_counterclockwise = vertices_counterclockwise * INEQ_CYCLE
values_siglocal = vertices_siglocal * INEQ_CYCLE

minimum_clockwise = minimum(values_clockwise)
minimum_counterclockwise = minimum(values_counterclockwise)
minimum_siglocal = minimum(values_siglocal)

saturating_clockwise = count(==(SIGLOCAL_BOUND), values_clockwise)
saturating_counterclockwise = count(==(SIGLOCAL_BOUND), values_counterclockwise)
saturating_indices = findall(==(SIGLOCAL_BOUND), values_siglocal)
saturating_siglocal = length(saturating_indices)

@assert minimum_clockwise == SIGLOCAL_BOUND
@assert minimum_counterclockwise == SIGLOCAL_BOUND
@assert minimum_siglocal == SIGLOCAL_BOUND
@assert all(values_siglocal .>= SIGLOCAL_BOUND)
@assert saturating_clockwise == 136
@assert saturating_counterclockwise == 136
@assert saturating_siglocal == 261

println("Exact Ineq_cycle audit")
println("  clockwise minimum:         ", minimum_clockwise)
println("  counterclockwise minimum:  ", minimum_counterclockwise)
println("  union minimum:             ", minimum_siglocal)
println("  saturating union vertices: ", saturating_siglocal)


Exact Ineq_cycle audit
  clockwise minimum:         20
  counterclockwise minimum:  20
  union minimum:             20
  saturating union vertices: 261


## 5. Exact facet test

A valid inequality defines a facet of a $d$-dimensional polytope exactly when
its saturating face has affine dimension $d-1$.

The routine below performs incremental Gaussian elimination over
`Rational{BigInt}`. It stores at most 64 basis rows, so it avoids constructing a
large dense rational matrix. This test is exact; no numerical rank tolerance is
used.


In [15]:
"""
    exact_affine_dimension(points; show_progress=false)

Compute affine dimension by exact row reduction over Rational{BigInt}. The
basis is maintained in reduced-row form and contains at most 64 rows.
"""
function exact_affine_dimension(points; show_progress=false)
    number_of_points, number_of_coordinates = size(points)
    number_of_points == 0 && return -1
    number_of_points == 1 && return 0

    reference = BigInt.(collect(@view points[1, :]))
    basis = Vector{Vector{Rational{BigInt}}}()
    pivots = Int[]

    for point_index in 2:number_of_points
        row = [
            (BigInt(points[point_index, column]) - reference[column]) // BigInt(1)
            for column in 1:number_of_coordinates
        ]

        # Eliminate all existing pivot columns from the new row.
        for basis_index in eachindex(basis)
            factor = row[pivots[basis_index]]
            if !iszero(factor)
                row .-= factor .* basis[basis_index]
            end
        end

        new_pivot = findfirst(value -> !iszero(value), row)
        if new_pivot !== nothing
            row ./= row[new_pivot]

            # Eliminate the new pivot from every existing basis row. Keeping
            # reduced-row form makes all later membership tests unambiguous.
            for basis_index in eachindex(basis)
                factor = basis[basis_index][new_pivot]
                if !iszero(factor)
                    basis[basis_index] .-= factor .* row
                end
            end

            insertion_index = searchsortedfirst(pivots, new_pivot)
            insert!(pivots, insertion_index, new_pivot)
            insert!(basis, insertion_index, row)
        end

        if show_progress && point_index % 1000 == 0
            println("  processed $point_index/$number_of_points points; rank = ", length(basis))
        end
    end

    return length(basis)
end

if RUN_EXACT_FACET_AUDIT
    println("Computing exact affine dimension of the full siglocal polytope...")
    siglocal_dimension = exact_affine_dimension(vertices_siglocal; show_progress=true)

    face_vertices = vertices_siglocal[saturating_indices, :]
    println("Computing exact affine dimension of the saturating face...")
    face_dimension = exact_affine_dimension(face_vertices)

    is_facet = face_dimension == siglocal_dimension - 1
    face_codimension = siglocal_dimension - face_dimension

    @assert siglocal_dimension == 50
    @assert face_dimension == 48
    @assert face_codimension == 2
    @assert !is_facet

    println("Exact facet audit")
    println("  polytope affine dimension: ", siglocal_dimension)
    println("  face affine dimension:     ", face_dimension)
    println("  face codimension:          ", face_codimension)
    println("  is a facet:                ", is_facet)
else
    println("Exact facet audit skipped by user configuration.")
end


Computing exact affine dimension of the full siglocal polytope...
  processed 1000/8128 points; rank = 30
  processed 2000/8128 points; rank = 37
  processed 3000/8128 points; rank = 44
  processed 4000/8128 points; rank = 44
  processed 5000/8128 points; rank = 49
  processed 6000/8128 points; rank = 50
  processed 7000/8128 points; rank = 50
  processed 8000/8128 points; rank = 50
Computing exact affine dimension of the saturating face...
Exact facet audit
  polytope affine dimension: 50
  face affine dimension:     48
  face codimension:          2
  is a facet:                false


## 6. A feasible quantum violation

This cell reconstructs the explicit three-qubit behavior recorded in Carlos's
notebook. All parties use the same two binary projective measurements. It is a
constructive quantum point, so a value below 20 proves that the inequality is
quantumly violated independently of NPA.

This feasible point need not be globally optimal. Its role is to give an upper
bound on $q_*$ and a lower bound on the maximum violation $v_*$.


In [17]:
function explicit_quantum_behaviour()
    ket0 = [1.0, 0.0]
    ket1 = [0.0, 1.0]

    projector0 = ket0 * ket0'
    projector1 = ket1 * ket1'

    alpha = pi / 2.8955
    phi_vector = cos(alpha) * ket0 + sin(alpha) * ket1
    projector_phi = phi_vector * phi_vector'
    identity2 = Matrix{Float64}(I, 2, 2)

    # E[(a,x)] is the projector for output a and input x. Bob and Charlie use
    # identical projectors, as in the recorded realization.
    E = Dict(
        (0, 0) => projector0,
        (1, 0) => projector1,
        (0, 1) => projector_phi,
        (1, 1) => identity2 - projector_phi,
    )

    W0 = (
        kron(ket1, ket0, ket0) +
        kron(ket0, ket1, ket0) +
        kron(ket0, ket0, ket1)
    ) / sqrt(3)

    W1 = (
        kron(ket0, ket1, ket1) +
        kron(ket1, ket0, ket1) +
        kron(ket1, ket1, ket0)
    ) / sqrt(3)

    theta = 0.041638
    state_angle = 5.78939
    w_angle = 5.182659

    psi = (
        cos(theta) * (
            cos(state_angle) * kron(ket0, ket0, ket0) +
            sin(state_angle) * kron(ket1, ket1, ket1)
        ) +
        sin(theta) * (cos(w_angle) * W0 + sin(w_angle) * W1)
    )

    @assert isapprox(norm(psi), 1.0; atol=1e-12, rtol=0.0)

    behaviour = zeros(Float64, 64)
    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        measurement = kron(E[(a, x)], E[(b, y)], E[(c, z)])
        probability = real(psi' * measurement * psi)
        behaviour[prob_index(a, b, c, x, y, z)] = probability
    end

    @assert minimum(behaviour) >= -1e-12
    for context in 0:7
        block = (8*context + 1):(8*context + 8)
        @assert isapprox(sum(behaviour[block]), 1.0; atol=1e-12, rtol=0.0)
    end

    return behaviour
end

EXPLICIT_BEHAVIOUR = explicit_quantum_behaviour()
EXPLICIT_VALUE = dot(Float64.(INEQ_CYCLE), EXPLICIT_BEHAVIOUR)
EXPLICIT_VIOLATION = SIGLOCAL_BOUND - EXPLICIT_VALUE

@assert EXPLICIT_VALUE < SIGLOCAL_BOUND
@assert isapprox(EXPLICIT_VALUE, 19.734369765878924; atol=1e-10, rtol=0.0)

@printf("Explicit quantum value I_cycle = %.15f\n", EXPLICIT_VALUE)
@printf("Explicit violation 20-I_cycle = %.15f\n", EXPLICIT_VIOLATION)


Explicit quantum value I_cycle = 19.734369765878913
Explicit violation 20-I_cycle = 0.265630234121087


## 7. Construct the Bell polynomial for `QuantumNPA.jl`

For each canonical coordinate, the probability operator is

$$
P^A_{a\mid x}P^B_{b\mid y}P^C_{c\mid z}.
$$

The coefficient vector is mapped automatically through `prob_index`; no
manually reordered NPA coefficient table is used.

The small compatibility method below is retained from the already tested NPA
notebook. It is installed only when the current `QuantumNPA`/`JuMP` combination
does not already provide the required sparse-matrix `dot` method.


In [19]:
function install_quantumnpa_jump_compatibility!()
    signature = Tuple{
        SparseMatrixCSC{Float64, Int},
        Symmetric{JuMP.VariableRef, Matrix{JuMP.VariableRef}},
    }

    if !isdefined(QuantumNPA, :dot) ||
       !hasmethod(getfield(QuantumNPA, :dot), signature)
        @eval QuantumNPA begin
            import JuMP: VariableRef
            import LinearAlgebra: Symmetric
            import SparseArrays: SparseMatrixCSC

            function dot(
                A::SparseMatrixCSC{Float64, Int},
                B::Symmetric{VariableRef, Matrix{VariableRef}},
            )
                return sum(
                    A[i, j] * B[j, i]
                    for i in 1:size(A, 1), j in 1:size(A, 2)
                )
            end
        end
        println("Installed the QuantumNPA/JuMP sparse-dot compatibility method.")
    else
        println("QuantumNPA/JuMP sparse-dot compatibility method already available.")
    end
end

install_quantumnpa_jump_compatibility!()

"""Create the 64 probability operators in exactly the canonical vector order."""
function build_probability_operators()
    PA = projector(1, 1:2, 1:2, full=true)
    PB = projector(2, 1:2, 1:2, full=true)
    PC = projector(3, 1:2, 1:2, full=true)

    operators = Vector{Any}(undef, 64)
    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operators[prob_index(a, b, c, x, y, z)] =
            PA[a + 1, x + 1] * PB[b + 1, y + 1] * PC[c + 1, z + 1]
    end
    return operators
end

"""Build the QuantumNPA polynomial from the exact 64 coefficients."""
function build_bell_expression(coefficients, probability_operators)
    length(coefficients) == length(probability_operators) == 64 ||
        error("A Bell expression requires exactly 64 coefficients/operators")

    expression = nothing
    for index in eachindex(coefficients)
        iszero(coefficients[index]) && continue
        term = Float64(coefficients[index]) * probability_operators[index]
        expression = expression === nothing ? term : expression + term
    end

    expression === nothing && error("The Bell functional is identically zero")
    return expression
end

PROBABILITY_OPERATORS = build_probability_operators()
BELL_EXPRESSION = build_bell_expression(INEQ_CYCLE, PROBABILITY_OPERATORS)

@assert length(PROBABILITY_OPERATORS) == 64
println("Bell expression constructed in canonical p(abc|xyz) order.")


QuantumNPA/JuMP sparse-dot compatibility method already available.
Bell expression constructed in canonical p(abc|xyz) order.


## 8. Tight NPA run

For each hierarchy level $k$, the notebook records

- $L_k$: the NPA lower bound on $q_*=\min_{\mathcal Q}I_{\rm cycle}$;
- $U_k=20-L_k$: the corresponding NPA upper bound on the maximum quantum
  violation $v_*$;
- elapsed wall-clock time.

The TSV file is rewritten after every successful level, so levels already
completed remain available if a later SDP is interrupted.


In [21]:
RESULTS_PATH = joinpath(RESULTS_DIRECTORY, "ineq_cycle_npa_bounds.tsv")

function write_results(path, rows)
    open(path, "w") do io
        println(io, join([
            "level",
            "npa_lower_bound_I_cycle",
            "npa_upper_bound_violation",
            "explicit_feasible_I_cycle",
            "explicit_achieved_violation",
            "elapsed_seconds",
        ], "\t"))

        for row in rows
            println(io, join([
                string(row.level),
                @sprintf("%.16g", row.lower_bound),
                @sprintf("%.16g", row.violation_upper_bound),
                @sprintf("%.16g", EXPLICIT_VALUE),
                @sprintf("%.16g", EXPLICIT_VIOLATION),
                @sprintf("%.6f", row.elapsed_seconds),
            ], "\t"))
        end
    end
end

npa_results = NamedTuple[]

for level in NPA_LEVELS
    println("Starting NPA level $level...")
    timed_result = @timed npa_min(
        BELL_EXPRESSION,
        level;
        solver=TIGHT_SCS_SOLVER,
    )

    lower_bound = Float64(timed_result.value)
    isfinite(lower_bound) || error("QuantumNPA returned non-finite value $lower_bound")

    # A valid NPA lower bound cannot exceed the value of a feasible quantum
    # point, apart from the declared numerical tolerance.
    lower_bound <= EXPLICIT_VALUE + NUMERICAL_TOLERANCE || error(
        "NPA lower bound $lower_bound exceeds the explicit feasible value " *
        "$EXPLICIT_VALUE by more than the numerical tolerance."
    )

    violation_upper_bound = SIGLOCAL_BOUND - lower_bound
    push!(npa_results, (
        level=level,
        lower_bound=lower_bound,
        violation_upper_bound=violation_upper_bound,
        elapsed_seconds=timed_result.time,
    ))

    write_results(RESULTS_PATH, npa_results)

    @printf(
        "  L_%d = %.12f;  violation upper bound = %.12f;  %.2f s\n",
        level,
        lower_bound,
        violation_upper_bound,
        timed_result.time,
    )

    GC.gc()
end

# For minimization, exact NPA bounds should be nondecreasing with the level.
# A small inversion is reported as a warning because SCS is numerical.
for index in 2:length(npa_results)
    previous = npa_results[index - 1]
    current = npa_results[index]
    if current.lower_bound < previous.lower_bound - NUMERICAL_TOLERANCE
        @warn "NPA lower bounds are not numerically monotone" previous current
    end
end

println("Saved NPA table to: ", abspath(RESULTS_PATH))


Starting NPA level 2...
  L_2 = 19.007235219340;  violation upper bound = 0.992764780660;  20.26 s
Starting NPA level 3...
  L_3 = 19.734320245849;  violation upper bound = 0.265679754151;  5.68 s
Starting NPA level 4...
  L_4 = 19.734319049306;  violation upper bound = 0.265680950694;  32.50 s
Saved NPA table to: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\ineq_cycle_npa_results\ineq_cycle_npa_bounds.tsv


## 9. Final certified interval and run metadata

If $L_4$ is the level-4 NPA result and $I_{\rm feasible}$ is the explicit
three-qubit value, then

$$
L_4\leq q_*\leq I_{\rm feasible},
$$

and equivalently

$$
20-I_{\rm feasible}\leq v_*\leq20-L_4.
$$

These are the two intervals printed below. The NPA side is a hierarchy bound;
the explicit side is a constructive quantum violation.


In [23]:
isempty(npa_results) && error("No NPA result was produced")
final_result = npa_results[end]

@printf("\nFinal interval using NPA level %d\n", final_result.level)
@printf(
    "  quantum minimum q*:      [%.12f, %.12f]\n",
    final_result.lower_bound,
    EXPLICIT_VALUE,
)
@printf(
    "  maximum violation v*:    [%.12f, %.12f]\n",
    EXPLICIT_VIOLATION,
    final_result.violation_upper_bound,
)

metadata_path = joinpath(RESULTS_DIRECTORY, "run_metadata.txt")
open(metadata_path, "w") do io
    println(io, "Ineq_cycle NPA run")
    println(io, "timestamp = ", now())
    println(io, "julia_version = ", VERSION)
    println(io, "npa_levels = ", join(NPA_LEVELS, ","))
    println(io, "scs_eps_abs = ", SCS_EPS_ABS)
    println(io, "scs_eps_rel = ", SCS_EPS_REL)
    println(io, "scs_max_iters = ", SCS_MAX_ITERS)
    println(io, "siglocal_bound = ", SIGLOCAL_BOUND)
    println(io, "explicit_value = ", @sprintf("%.16g", EXPLICIT_VALUE))
    println(io, "explicit_violation = ", @sprintf("%.16g", EXPLICIT_VIOLATION))
    println(io, "final_npa_level = ", final_result.level)
    println(io, "final_npa_lower_bound_I = ", @sprintf("%.16g", final_result.lower_bound))
    println(io, "final_npa_upper_bound_violation = ", @sprintf("%.16g", final_result.violation_upper_bound))
    if RUN_EXACT_FACET_AUDIT
        println(io, "siglocal_affine_dimension = ", siglocal_dimension)
        println(io, "saturating_face_affine_dimension = ", face_dimension)
        println(io, "saturating_vertices = ", saturating_siglocal)
        println(io, "is_facet = ", is_facet)
    end
end

println("Saved run metadata to: ", abspath(metadata_path))



Final interval using NPA level 4
  quantum minimum q*:      [19.734319049306, 19.734369765879]
  maximum violation v*:    [0.265630234121, 0.265680950694]
Saved run metadata to: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\ineq_cycle_npa_results\run_metadata.txt


## Interpretation checklist

- The exact classical test must print a minimum of `20` in both orientations
  and in their union.
- The exact facet test must print dimensions `50` (polytope) and `48` (face),
  hence `is a facet: false`.
- The explicit realization must print approximately
  `19.734369765878924 < 20`; this alone establishes a quantum violation.
- The most useful NPA quantity for this lower-bound inequality is the last
  column conceptually represented by $20-L_k$: it is an **upper bound on the
  maximum violation**.
- Send back `ineq_cycle_npa_bounds.tsv`, `run_metadata.txt`, and the final
  printed interval for validation.
